In [1]:
! pip install openai


In [2]:
import os
from openai import OpenAI

In [5]:
from openai import OpenAI
from google.colab import userdata

# Get the Groq API key from Colab's secrets manager
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Ensure the API key is not None
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    input="Explain the importance of fast language models",
    model="openai/gpt-oss-20b",
)
print(response.output_text)

### Why “fast” matters for language models

| Domain | Why speed is critical | Typical impact |
|--------|-----------------------|----------------|
| **Real‑time user experiences** (chatbots, voice assistants, instant translation) | Users expect a response in *milliseconds*, not seconds. High latency breaks the flow of conversation and can turn a good product into a frustrating one. | Lower abandonment rates, higher engagement, better brand perception. |
| **Edge & mobile deployment** | Devices often lack GPUs or have limited battery. Fast inference means the model can run on-device (privacy‑friendly) without relying on a cloud backend. | Enables offline AI, reduces data usage, opens AI to low‑bandwidth regions. |
| **High‑throughput workloads** (search engines, recommendation engines, content moderation) | A single model may serve millions of requests per day. Even a 10‑ms slowdown per request can translate into minutes of extra compute time each hour. | Cost savings in cloud billing,

In [6]:
! pip install langchain faiss-cpu langchain_community tiktoken sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is i

In [7]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/company_dataset_additional.txt')
documents=loader.load()

/tmp/ipykernel_1548/1587967044.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
# step 2 create the embedding + vector DB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the HuggingFace embeddings
# Using a common open-source model; ensure 'sentence-transformers' is installed
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Create the vector database from your pre-loaded documents
vector_db = FAISS.from_documents(documents, embeddings)

/tmp/ipykernel_1548/4207546490.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# #step 3 Retrieval + Generation
def rag_query(query):
    # Retrieve the top 3 most relevant document chunks based on the query
    docs = vector_db.similarity_search(query, k=3)

    # Combine the content of the retrieved documents into a single context string
    context = " ".join([doc.page_content for doc in docs])

    # Generate the response using OpenAI's chat completions API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model to the user-specified Groq model
        messages=[
            {"role": "system", "content": "use the provided context to answer the question"},
            {"role": "user", "content": f"context: {context}\n\nquery: {query}"}
        ]
    )

    return response.choices[0].message.content

In [10]:
print(rag_query("What is the refund policy of SkyTech Electronics Pvt Ltd?"))

The refund policy of SkyTech Electronics Pvt Ltd is as follows: 

Customers can request a refund within 15 days of delivery. Refunds are processed within 5-8 business days after approval.


In [11]:
import warnings
warnings.filterwarnings('ignore')
!pip install bitsandbytes accelerate transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00


In [12]:
# #step 1 : Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# #define the bitsandbytes config for 8 bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [13]:
# #step 2 : Apply LoRA(PEFT)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# #preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)


In [14]:
from datasets import Dataset

data = [
    {
        "text": "Q: What is ABC Retail Pvt Ltd?\nA: ABC Retail is an e-commerce company specializing in electronics, fashion, and home appliances."
    },

    {
        "text": "Q: What is the refund policy of ABC Retail?\nA: Customers can request a refund within 7 days of delivery. Refunds are processed within 5-7 business days after approval."
    },

    {
        "text": "Q: What is the return policy of ABC Retail?\nA: Products can be returned within 10 days if unused and in original packaging."
    },

    {
        "text": "Q: What are the shipping options available at ABC Retail?\nA: Standard delivery takes 3-5 business days and express delivery takes 1-2 business days."
    },

    {
        "text": "Q: How can I contact ABC Retail customer support?\nA: Support is available 24/7 via email and phone. Email: support@abcretail.com, Phone: +91-9876543210."
    },

    {
        "text": "Q: What are the business hours of ABC Retail?\nA: Monday to Saturday: 9 AM to 8 PM. Sunday: Closed."
    },

    {
        "text": "Q: Does ABC Retail protect customer data?\nA: Yes, customer data is encrypted and not shared with third parties without consent."
    },

    {
        "text": "Q: Does ABC Retail have a loyalty program?\nA: Yes, customers earn points on every purchase which can be redeemed for discounts."
    },

    {
        "text": "Q: How long does ABC Retail take to process refunds?\nA: Refunds are processed within 5-7 business days after approval."
    },

    {
        "text": "Q: Can I return a used product to ABC Retail?\nA: No, products can only be returned if they are unused and in their original packaging."
    },

    {
        "text": "Q: What is SkyTech Electronics?\nA: SkyTech Electronics is an online retailer specializing in smartphones, laptops, accessories, and smart home devices."
    },

    {
        "text": "Q: What is the refund policy of SkyTech Electronics?\nA: Customers can request a refund within 15 days of delivery. Refunds are processed within 5-8 business days after approval."
    },

    {
        "text": "Q: What is the return policy of SkyTech Electronics?\nA: Products can be returned within 20 days if unused and in original packaging."
    },

    {
        "text": "Q: What are the shipping options available at SkyTech Electronics?\nA: Standard delivery takes 4-6 business days and express delivery takes 1-2 business days."
    },

    {
        "text": "Q: How can I contact SkyTech Electronics customer support?\nA: Support is available 24/7 via email, chat, and phone. Email: support@skytech.com, Phone: +91-9876501001."
    },

    {
        "text": "Q: What are the business hours of SkyTech Electronics?\nA: Monday to Saturday: 9 AM to 9 PM. Sunday: 10 AM to 6 PM."
    },

    {
        "text": "Q: What is FreshBasket Grocery?\nA: FreshBasket Grocery delivers fresh fruits, vegetables, dairy products, and daily essentials to customers' doorsteps."
    },

    {
        "text": "Q: What is the refund policy of FreshBasket Grocery?\nA: Refund requests are accepted within 3 days of delivery. Refunds are processed within 2-4 business days."
    },

    {
        "text": "Q: Can I return perishable items purchased from FreshBasket Grocery?\nA: Perishable items can only be returned if damaged upon delivery."
    },

    {
        "text": "Q: How long does FreshBasket take for delivery?\nA: Same-day delivery is available in selected cities, while standard delivery takes 1-2 business days."
    },

    {
        "text": "Q: How can I contact FreshBasket Grocery customer support?\nA: Support is available from 6 AM to 11 PM. Email: care@freshbasket.com, Phone: +91-9876501002."
    },

    {
        "text": "Q: Does FreshBasket Grocery have a loyalty program?\nA: Yes, customers earn FreshRewards points on every order."
    }
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 22
})


In [15]:
# step 4: tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True,padding="max_length",
    max_length=128)


tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

In [16]:
from transformers import Trainer , TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)

# Defensive check and re-application of PEFT if model is not recognized as a PeftModel
# This ensures that the model is correctly prepared for PEFT training
if not isinstance(model, PeftModel):
    print("Warning: Model not recognized as a PEFT model. Attempting to re-apply PEFT configuration.")
    # Redefine lora_config to ensure it's available in this scope
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.5,
        bias="none",
        task_type="CAUSAL_LM"
    )
    # The 'model' variable is assumed to be the base quantized model at this point if it's not a PeftModel.
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print("PEFT configuration re-applied.")

training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Step,Training Loss
10,7.045609
20,3.692627
30,2.601292


TrainOutput(global_step=30, training_loss=4.446509170532226, metrics={'train_runtime': 60.3005, 'train_samples_per_second': 1.824, 'train_steps_per_second': 0.498, 'total_flos': 87490764472320.0, 'train_loss': 4.446509170532226, 'epoch': 5.0})

In [17]:
def generate_response(question):
    prompt = f"Q: {question}\nA:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [18]:
print(generate_response("What is ABC Retail Pvt Ltd?"))

print(generate_response("What is the refund policy of ABC Retail?"))

print(generate_response("What is the return policy of ABC Retail?"))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is ABC Retail Pvt Ltd?
A: ABC, and the same time.













































[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the refund policy of ABC Retail?
A: ABC, and the same time.











































Q: What is the return policy of ABC Retail?
A: ABC, and the same time.











































